# Create Arcadia Fund Awards

Creates OpenAlex award rows from Arcadia Fund's official 360Giving grants CSV.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/arcadia_to_s3.py` to discover the current CSV from Arcadia's grants page, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://arcadiafund.org.uk/grants-awarded`, which links to Arcadia's official 360Giving CSV.  
**S3 location:** `s3a://openalex-ingest/awards/arcadia/arcadia_grants.parquet`  
**Local full-source validation on 2026-05-27:** 498 official grant rows from the current `Arcadia-360G-data-May-2026.csv`; 100% native ID/title/description/recipient/recipient URL/year/duration/amount/currency coverage; year range 2002-2025; total USD 1,333,778,093.82. Funding areas: Open Access 152, Culture 134, Nature 133, Discretionary 79.

**OpenAlex funder mapping:**
- funder_id: 4320313262
- display_name: `Arcadia Fund`
- ROR: NULL in OpenAlex API
- DOI: NULL in OpenAlex API

**Mapping summary:** One OpenAlex award row per 360Giving `Identifier`. `amount` and `currency` come directly from source `Amount Awarded` and `Currency` fields. `funder_scheme` is `Grant Programme: Title`, with `Grant Programme: Subtitle` retained in raw staging as `grant_priority` for audit.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arcadia_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/arcadia/arcadia_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 498 rows.
SELECT COUNT(*) as total_arcadia_grants
FROM openalex.awards.arcadia_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.arcadia_raw;


In [ ]:
%sql
-- Sample raw Arcadia data.
SELECT
    funder_award_id,
    display_name,
    beneficiary,
    funding_area,
    grant_priority,
    amount,
    currency,
    source_year,
    duration_years,
    landing_page_url
FROM openalex.awards.arcadia_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'arcadia_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|usd|currency';


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount_usd,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.arcadia_raw;


In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_row_hash) AS distinct_source_rows
FROM openalex.awards.arcadia_raw;


In [ ]:
%sql
-- Funding-area and status distribution.
SELECT funding_area, source_status, COUNT(*) AS cnt, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_usd
FROM openalex.awards.arcadia_raw
GROUP BY funding_area, source_status
ORDER BY cnt DESC;


In [ ]:
%sql
-- Coverage by major source fields.
SELECT
    COUNT(*) AS total,
    COUNT(description) AS has_description,
    COUNT(beneficiary) AS has_beneficiary,
    COUNT(beneficiary_url) AS has_beneficiary_url,
    COUNT(funding_area) AS has_funding_area,
    COUNT(grant_priority) AS has_grant_priority,
    COUNT(recipient_org_identifier) AS has_recipient_org_identifier,
    COUNT(recipient_org_charity_number) AS has_charity_number
FROM openalex.awards.arcadia_raw;


## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320313262;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arcadia_awards
USING delta
AS
WITH
arcadia_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320313262
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN NULLIF(TRIM(currency), '') ELSE NULL END AS parsed_currency,
        COALESCE(TRY_TO_DATE(award_date, 'yyyy-MM-dd'), TRY_TO_DATE(start_date, 'yyyy-MM-dd')) AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_start_year,
        TRY_CAST(duration_years AS INT) AS parsed_duration_years
    FROM openalex.awards.arcadia_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        g.parsed_currency as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        'grant' as funding_type,
        COALESCE(NULLIF(TRIM(g.funding_area), ''), 'Arcadia grant') as funder_scheme,
        'arcadia_360giving' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_start_year + g.parsed_duration_years - 1) as end_year,
        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.beneficiary), '') as name,
                CAST(NULL AS STRING) as country,
                CASE
                    WHEN g.recipient_org_identifier IS NOT NULL AND TRIM(g.recipient_org_identifier) != '' THEN
                        ARRAY(struct(TRIM(g.recipient_org_identifier) as id, '360giving_recipient_org' as type, 'source' as asserted_by))
                    ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                END as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS STRING)) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN arcadia_funder f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 148

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'arcadia_360giving' AND priority = 148;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    148 as priority
FROM openalex.awards.arcadia_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_arcadia_awards
FROM openalex.awards.arcadia_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.arcadia_awards;


In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date, end_date, lead_investigator.affiliation.name AS beneficiary, landing_page_url
FROM openalex.awards.arcadia_awards
LIMIT 10;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids, COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.arcadia_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.arcadia_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.affiliation.name) as has_beneficiary,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(SUM(amount), 2) as total_funding
FROM openalex.awards.arcadia_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount,
    ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.arcadia_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 2) as total_usd
FROM openalex.awards.arcadia_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 2) as total_usd
FROM openalex.awards.arcadia_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids, ROUND(SUM(amount), 2) as total_usd
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'arcadia_360giving' AND priority = 148
GROUP BY priority, provenance;


## Admin Handoff Notes

Contractor has no S3/Databricks access. Admin must upload the parquet with `scripts/local/arcadia_to_s3.py`, run this notebook on Databricks, inspect the Step 6 verification cells, then run the combined CreateAwards notebook. Do not add job YAML until the Databricks ingest is run and QA is complete.
